Import and loading data

In [49]:
import pandas as pd
import joblib

dataset_2022 = pd.read_csv("../data/2022 grandprix abu dhabi.csv")
dataset_2023 = pd.read_csv("../data/2023 grandprix abu dhabi.csv")
dataset_2024 = pd.read_csv("../data/2024 grandprix abu dhabi.csv")
dataset_2025 = pd.read_csv("../data/2025-Abu Dhabi Grand Prix-Race.csv")

In [50]:
dataset_2022["Race"] = "2022"
dataset_2023["Race"] = "2023"
dataset_2024["Race"] = "2024"
dataset_2025["Race"] = "2025"

combining all the datasets

In [51]:
dataset = pd.concat([dataset_2022, dataset_2023, dataset_2024, dataset_2025], ignore_index=True)

keeping only useful columns

In [52]:
useful_columns = [
    "LapNumber",
    "Stint",
    "Compound",
    "TyreLife",
    "Team",
    "LapTime_in_seconds",
    "Race"
]

clean_dataset = dataset[useful_columns].copy()

#clean_dataset.head()

removing rows with missing values

In [53]:
clean_dataset = clean_dataset.dropna()
#clean_dataset.shape

ensuring all numeric columns are actually numeric

In [54]:
numeric_columns = ["LapNumber",
                   "Stint",
                   "TyreLife",
                   "LapTime_in_seconds"]

clean_dataset[numeric_columns] = clean_dataset[numeric_columns].apply(pd.to_numeric, errors = "coerce")

removing the outliers (slow laps, pit laps, yellow flag laps)

In [55]:
median = clean_dataset["LapTime_in_seconds"].median()
clean_dataset = clean_dataset[clean_dataset["LapTime_in_seconds"] < 1.2 * median]
#clean_dataset.shape

encoding catergorial values

In [56]:
# creating mappings (rules) that assign unique number to each tyre compound and team
# (this does not change the dataset yet, it only defines how values will be converted)
compound_map = {c: i for i, c in enumerate(clean_dataset["Compound"].unique())}
team_map = {t: i for i, t in enumerate(clean_dataset["Team"].unique())}

# applying mappings to datasets, replacing the text values with their corresponding numbers
clean_dataset["Compound"] = clean_dataset["Compound"].map(compound_map)
clean_dataset["Team"] = clean_dataset["Team"].map(team_map)

#clean_dataset.head()

Adding and approximating fuel load

In [57]:
clean_dataset["FuelLoad"] = clean_dataset.groupby("Race")["LapNumber"] \
    .transform(lambda x: x.max() - x)

#clean_dataset.head()

save cleaned data and mappings 

In [58]:
clean_dataset.to_csv("../data/clean_dataset.csv", index=False)

joblib.dump(compound_map, "../models/compound_map.pkl")
joblib.dump(team_map, "../models/team_map.pkl")

['../models/team_map.pkl']

In [59]:
print(clean_dataset[clean_dataset["TyreLife"] <= 2][["TyreLife", "LapTime_in_seconds"]].groupby("TyreLife").mean())

          LapTime_in_seconds
TyreLife                    
1.0                98.751436
2.0                90.767482
